In [1]:
!pip install mediapipe opencv-python scikit-learn numpy matplotlib tqdm fastapi uvicorn pydantic torch --quiet
print('\n✅ All packages installed!')



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

✅ All packages installed!


In [2]:
import os
import cv2
import time
import numpy as np

# ════════════════════════════════════════════
# ✏️  CONFIGURE THESE
WORDS              = [
    'hello', 'thank_you', 'please', 'you',
    'yes', 'no', 'help', 'home', 'friend', 'i_love_you'
]
SAMPLES_PER_WORD   = 30      # how many video clips to record per word
FRAMES_PER_SAMPLE  = 60      # frames per clip  (~2 sec at 30 fps)
DATASET_DIR        = 'dataset'
CAM_INDEX          = 0       # change to 1 if external webcam
# ════════════════════════════════════════════

os.makedirs(DATASET_DIR, exist_ok=True)
for w in WORDS:
    os.makedirs(os.path.join(DATASET_DIR, w), exist_ok=True)

cap = cv2.VideoCapture(CAM_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# ── Helpers ───────────────────────────────────────────────────────────────
def overlay(frame, lines, y_start=40, color=(255, 255, 255), scale=0.7, thickness=2):
    for i, line in enumerate(lines):
        cv2.putText(frame, line,
                    (20, y_start + i * 35),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness, cv2.LINE_AA)

def count_existing(word):
    d = os.path.join(DATASET_DIR, word)
    return len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])

# ── Main collection loop ──────────────────────────────────────────────────
word_idx = 0
print("OpenCV window opened. Follow the instructions on screen.")
print("Press S to record, N for next word, Q to quit.\n")

while word_idx < len(WORDS):
    word    = WORDS[word_idx]
    done    = count_existing(word)
    remaining = SAMPLES_PER_WORD - done

    if remaining <= 0:
        print(f'  ✅ {word} already has {done} samples — skipping')
        word_idx += 1
        continue

    # ── WAITING state ─────────────────────────────────────────────────────
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)   # mirror
        overlay(frame, [
            f'Word  :  {word.upper()}  ({word_idx+1}/{len(WORDS)})',
            f'Collected:  {done} / {SAMPLES_PER_WORD}',
            '',
            'Press  S  to record this sample',
            'Press  N  to skip to next word',
            'Press  Q  to quit',
        ], y_start=50, color=(50, 220, 50))

        cv2.imshow('ISL Word Collector', frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('s'):
            break
        elif key == ord('n'):
            word_idx += 1
            break
        elif key == ord('q'):
            word_idx = len(WORDS)   # exit outer loop
            break

    if word_idx >= len(WORDS) or (remaining <= 0):
        break
    if cv2.waitKey(1) & 0xFF == ord('n'):
        continue

    # ── COUNTDOWN 3-2-1 ───────────────────────────────────────────────────
    for cnt in [3, 2, 1]:
        deadline = time.time() + 1.0
        while time.time() < deadline:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            overlay(frame, [f'Starting in  {cnt}...'], y_start=220,
                    color=(0, 200, 255), scale=1.4, thickness=3)
            cv2.imshow('ISL Word Collector', frame)
            cv2.waitKey(1)

    # ── RECORD ────────────────────────────────────────────────────────────
    sample_idx  = done
    sample_dir  = os.path.join(DATASET_DIR, word, f'sample_{sample_idx:03d}')
    os.makedirs(sample_dir, exist_ok=True)

    frames_saved = 0
    while frames_saved < FRAMES_PER_SAMPLE:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)

        # Save frame
        cv2.imwrite(os.path.join(sample_dir, f'frame_{frames_saved:03d}.jpg'), frame)
        frames_saved += 1

        # HUD
        progress = int((frames_saved / FRAMES_PER_SAMPLE) * 200)
        cv2.rectangle(frame, (20, 440), (20 + progress, 460), (0, 140, 255), -1)
        cv2.rectangle(frame, (20, 440), (220, 460), (200, 200, 200), 2)
        overlay(frame, [
            f'RECORDING  {word.upper()}',
            f'Frame  {frames_saved} / {FRAMES_PER_SAMPLE}',
        ], y_start=50, color=(0, 80, 255), scale=0.9, thickness=2)
        cv2.imshow('ISL Word Collector', frame)
        cv2.waitKey(1)

    done += 1
    remaining -= 1
    print(f'  Saved: {word}/sample_{sample_idx:03d}  ({done}/{SAMPLES_PER_WORD})')

    # ── PAUSE between samples ─────────────────────────────────────────────
    if remaining > 0:
        pause_end = time.time() + 1.5
        while time.time() < pause_end:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            overlay(frame, [
                f'✅  Sample {done} saved!',
                f'Reset hand — next in 1.5s ...',
            ], y_start=200, color=(0, 220, 120), scale=0.85)
            cv2.imshow('ISL Word Collector', frame)
            cv2.waitKey(1)
    else:
        word_idx += 1   # move to next word automatically when done

cap.release()
cv2.destroyAllWindows()
print('\n✅ Data collection complete!')
for w in WORDS:
    n = count_existing(w)
    print(f'   {w:15s}: {n} samples')


OpenCV window opened. Follow the instructions on screen.
Press S to record, N for next word, Q to quit.

  Saved: hello/sample_000  (1/30)
  Saved: hello/sample_001  (2/30)
  Saved: hello/sample_002  (3/30)
  Saved: hello/sample_003  (4/30)
  Saved: hello/sample_004  (5/30)
  Saved: hello/sample_005  (6/30)
  Saved: hello/sample_006  (7/30)
  Saved: hello/sample_007  (8/30)
  Saved: hello/sample_008  (9/30)
  Saved: hello/sample_009  (10/30)
  Saved: hello/sample_010  (11/30)
  Saved: hello/sample_011  (12/30)
  Saved: hello/sample_012  (13/30)
  Saved: hello/sample_013  (14/30)
  Saved: hello/sample_014  (15/30)
  Saved: hello/sample_015  (16/30)
  Saved: hello/sample_016  (17/30)
  Saved: hello/sample_017  (18/30)
  Saved: hello/sample_018  (19/30)
  Saved: hello/sample_019  (20/30)
  Saved: hello/sample_020  (21/30)
  Saved: hello/sample_021  (22/30)
  Saved: hello/sample_022  (23/30)
  Saved: hello/sample_023  (24/30)
  Saved: hello/sample_024  (25/30)
  Saved: hello/sample_025  (2

In [3]:
import os, urllib.request
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm import tqdm

# ════════════════════════════════════════════
MAX_FRAMES  = 60          # must match FRAMES_PER_SAMPLE above
DATASET_DIR = 'dataset'
OUTPUT_PATH = 'data/word_keypoints.npz'
IMG_EXTS    = ('.jpg', '.jpeg', '.png', '.bmp')
# ════════════════════════════════════════════

os.makedirs('data', exist_ok=True)

# ── Download hand landmarker model ────────────────────────────────────────
MODEL_PATH = 'hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Downloading hand landmarker model (~9MB)...')
    url = ('https://storage.googleapis.com/mediapipe-models/'
           'hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task')
    urllib.request.urlretrieve(url, MODEL_PATH)
    print('Downloaded:', MODEL_PATH)
else:
    print('Model file already exists:', MODEL_PATH)

# ── Build detector ────────────────────────────────────────────────────────
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.4,
    min_hand_presence_confidence=0.4,
    min_tracking_confidence=0.4,
    running_mode=mp_vision.RunningMode.IMAGE,
)
detector = mp_vision.HandLandmarker.create_from_options(options)

# ── Helpers ───────────────────────────────────────────────────────────────
def normalize(kps):
    pts   = kps.reshape(21, 3)
    pts   = pts - pts[0]             # wrist to origin
    scale = np.linalg.norm(pts[9])   # middle-finger MCP distance
    if scale > 0:
        pts /= scale
    return pts.flatten()             # (63,)

def detect_frame(img_bgr):
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result   = detector.detect(mp_image)
    if not result.hand_landmarks:
        return None
    lms = result.hand_landmarks[0]
    kps = np.array([[lm.x, lm.y, lm.z] for lm in lms], dtype=np.float32).flatten()
    return normalize(kps)

def pad_or_truncate(seq, max_frames):
    arr = np.array(seq, dtype=np.float32)   # (T, 63)
    if len(arr) >= max_frames:
        return arr[:max_frames]
    pad = np.zeros((max_frames - len(arr), 63), dtype=np.float32)
    return np.concatenate([arr, pad], axis=0)

# ── Main extraction loop ──────────────────────────────────────────────────
X, y = [], []
failed = 0

words = sorted([d for d in os.listdir(DATASET_DIR)
                if os.path.isdir(os.path.join(DATASET_DIR, d))])

for word in words:
    word_dir  = os.path.join(DATASET_DIR, word)
    samples   = sorted([s for s in os.listdir(word_dir)
                        if os.path.isdir(os.path.join(word_dir, s))])
    extracted = 0

    for sample in tqdm(samples, desc=f'{word} ({len(samples)} samples)', leave=False):
        sample_dir = os.path.join(word_dir, sample)
        frames_sorted = sorted([f for f in os.listdir(sample_dir)
                                 if f.lower().endswith(IMG_EXTS)])

        seq = []
        for fname in frames_sorted:
            img = cv2.imread(os.path.join(sample_dir, fname))
            if img is None:
                continue
            kp = detect_frame(img)
            if kp is not None:
                seq.append(kp)

        if len(seq) < 5:     # too few landmarks detected → skip
            failed += 1
            continue

        padded = pad_or_truncate(seq, MAX_FRAMES)  # (MAX_FRAMES, 63)
        X.append(padded)
        y.append(word)
        extracted += 1

    print(f'  {word}: {extracted}/{len(samples)} extracted')

detector.close()

X = np.array(X, dtype=np.float32)   # (N, MAX_FRAMES, 63)
y = np.array(y)
np.savez_compressed(OUTPUT_PATH, X=X, y=y)

print(f'\n✅ Saved {len(X)} samples → {OUTPUT_PATH}')
print(f'   Failed (< 5 frames detected) : {failed}')
print(f'   Classes  : {sorted(set(y))}')
print(f'   Shape    : X={X.shape}  y={y.shape}')


Downloaded: hand_landmarker.task


I0000 00:00:1777966814.039946 13146441 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
I0000 00:00:1777966814.098591 13146441 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777966814.106485 13146446 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777966814.116617 13146445 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
friend (16 samples):   0%|          | 0/16 [00:00<?, ?it/s]W0000 00:00:1777966814.195097 13146446 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


  friend: 16/16 extracted


  hello: 30/30 extracted


  help: 9/11 extracted


...

yes (30 samples):  10%|█         | 3/30 [00:04<00:45,  1.68s/it]E0000 00:00:1777967114.385614 13146442 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-05T13:26:14.374626+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


  yes: 29/30 extracted


you (30 samples):  33%|███▎      | 10/30 [00:15<00:33,  1.66s/it]E0000 00:00:1777967174.394144 13146442 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-05T13:26:14.374626+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
                                                                 

  you: 29/30 extracted

✅ Saved 236 samples → data/word_keypoints.npz
   Failed (< 5 frames detected) : 4
   Classes  : [np.str_('friend'), np.str_('hello'), np.str_('help'), np.str_('home'), np.str_('i_love_you'), np.str_('no'), np.str_('please'), np.str_('thank_you'), np.str_('yes'), np.str_('you')]
   Shape    : X=(236, 60, 63)  y=(236,)


E0000 00:00:1777967207.252344 13146442 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-05T13:26:14.374626+05:30
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


In [1]:
import os, pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

os.makedirs('models', exist_ok=True)

print('Loading word keypoints...')
data = np.load('data/word_keypoints.npz')
X, y = data['X'], data['y']           # X: (N, T, 63)
MAX_FRAMES = X.shape[1]

print(f'  Samples  : {len(X)}')
print(f'  Sequence : {X.shape[1]} frames × {X.shape[2]} features')
print(f'  Classes  : {sorted(set(y))}')

# Encode labels
le    = LabelEncoder()
y_enc = le.fit_transform(y)
NUM_CLASSES = len(le.classes_)
print(f'\nLabel map: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Flat features (MLP / SVM)
X_flat = X.reshape(len(X), -1)
X_tr_f, X_te_f, y_tr, y_te = train_test_split(
    X_flat, y_enc, test_size=0.15, stratify=y_enc, random_state=42)

# Sequence split (same indices for LSTM)
all_idx = np.arange(len(X))
_, idx_te = train_test_split(all_idx, test_size=0.15, stratify=y_enc, random_state=42)
idx_tr    = np.setdiff1d(all_idx, idx_te)
X_seq_tr, y_seq_tr = X[idx_tr], y_enc[idx_tr]
X_seq_te, y_seq_te = X[idx_te], y_enc[idx_te]

print(f'\nTrain: {len(X_tr_f)} | Test: {len(X_te_f)}')


Loading word keypoints...
  Samples  : 236
  Sequence : 60 frames × 63 features
  Classes  : [np.str_('friend'), np.str_('hello'), np.str_('help'), np.str_('home'), np.str_('i_love_you'), np.str_('no'), np.str_('please'), np.str_('thank_you'), np.str_('yes'), np.str_('you')]

Label map: {np.str_('friend'): np.int64(0), np.str_('hello'): np.int64(1), np.str_('help'): np.int64(2), np.str_('home'): np.int64(3), np.str_('i_love_you'): np.int64(4), np.str_('no'): np.int64(5), np.str_('please'): np.int64(6), np.str_('thank_you'): np.int64(7), np.str_('yes'): np.int64(8), np.str_('you'): np.int64(9)}

Train: 200 | Test: 36


In [2]:
# ── Train MLP ─────────────────────────────────────────────────────────────
print('Training MLP...')
mlp_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=(512, 256, 128),
        activation='relu', solver='adam', alpha=1e-4,
        batch_size=32, learning_rate='adaptive', max_iter=500,
        random_state=42, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=20, verbose=False,
    ))
])
mlp_pipe.fit(X_tr_f, y_tr)
mlp_preds = mlp_pipe.predict(X_te_f)
mlp_acc   = accuracy_score(y_te, mlp_preds)
print(f'✅ MLP Test Accuracy: {mlp_acc*100:.1f}%')


Training MLP...
✅ MLP Test Accuracy: 83.3%


In [3]:
# ── Train SVM ─────────────────────────────────────────────────────────────
print('Training SVM (RBF)...')
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42))
])
svm_pipe.fit(X_tr_f, y_tr)
svm_preds = svm_pipe.predict(X_te_f)
svm_acc   = accuracy_score(y_te, svm_preds)
print(f'✅ SVM Test Accuracy: {svm_acc*100:.1f}%')


Training SVM (RBF)...
✅ SVM Test Accuracy: 94.4%


In [4]:
# ── Train LSTM (Bidirectional + Attention) ─────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training LSTM on {device}...')

EPOCHS = 60
BATCH  = 16
LR     = 1e-3

class WordLSTM(nn.Module):
    def __init__(self, input_dim=63, hidden=256, layers=2, num_classes=10, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.attn = nn.Linear(hidden * 2, 1)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        out, _ = self.lstm(x)                       # (B, T, H*2)
        w      = torch.softmax(self.attn(out), dim=1)  # (B, T, 1)
        ctx    = (w * out).sum(dim=1)               # (B, H*2)
        return self.head(ctx)

Xtr_t = torch.tensor(X_seq_tr, dtype=torch.float32)
ytr_t = torch.tensor(y_seq_tr, dtype=torch.long)
Xte_t = torch.tensor(X_seq_te, dtype=torch.float32)
yte_t = torch.tensor(y_seq_te, dtype=torch.long)

train_dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=BATCH, shuffle=True)
model    = WordLSTM(num_classes=NUM_CLASSES).to(device)
opt      = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched    = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)
loss_fn  = nn.CrossEntropyLoss()

best_lstm_acc, best_state = 0.0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss_fn(model(xb), yb).backward()
        opt.step()
    sched.step()

    if epoch % 10 == 0 or epoch == EPOCHS:
        model.eval()
        with torch.no_grad():
            preds_e = model(Xte_t.to(device)).argmax(1).cpu().numpy()
        acc_e = accuracy_score(y_seq_te, preds_e)
        print(f'  Epoch {epoch:3d}/{EPOCHS}  |  val acc: {acc_e*100:.1f}%')
        if acc_e > best_lstm_acc:
            best_lstm_acc = acc_e
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    lstm_preds = model(Xte_t.to(device)).argmax(1).cpu().numpy()
lstm_acc = accuracy_score(y_seq_te, lstm_preds)
print(f'\n✅ LSTM Best Test Accuracy: {lstm_acc*100:.1f}%')


Training LSTM on cpu...
  Epoch  10/60  |  val acc: 97.2%
  Epoch  20/60  |  val acc: 91.7%
  Epoch  30/60  |  val acc: 100.0%
  Epoch  40/60  |  val acc: 100.0%
  Epoch  50/60  |  val acc: 100.0%
  Epoch  60/60  |  val acc: 100.0%

✅ LSTM Best Test Accuracy: 100.0%


In [5]:
# ── Pick winner ───────────────────────────────────────────────────────────
scores = {
    'MLP':  (mlp_acc,  mlp_preds,  y_te),
    'SVM':  (svm_acc,  svm_preds,  y_te),
    'LSTM': (lstm_acc, lstm_preds, y_seq_te),
}
best_name, (best_acc, best_preds, best_y_te) = max(scores.items(), key=lambda kv: kv[1][0])
print(f'🏆  Best model : {best_name}  ({best_acc*100:.1f}%)')
print(f'    MLP  : {mlp_acc*100:.1f}%    SVM  : {svm_acc*100:.1f}%    LSTM : {lstm_acc*100:.1f}%')

report = classification_report(best_y_te, best_preds, target_names=le.classes_)
print('\n', report)

with open('models/word_report.txt', 'w') as f:
    for name, (acc, _, _) in scores.items():
        f.write(f'{name} accuracy: {acc*100:.2f}%\n')
    f.write('\n' + report)


🏆  Best model : LSTM  (100.0%)
    MLP  : 83.3%    SVM  : 94.4%    LSTM : 100.0%

               precision    recall  f1-score   support

      friend       1.00      1.00      1.00         2
       hello       1.00      1.00      1.00         5
        help       1.00      1.00      1.00         1
        home       1.00      1.00      1.00         3
  i_love_you       1.00      1.00      1.00         2
          no       1.00      1.00      1.00         5
      please       1.00      1.00      1.00         5
   thank_you       1.00      1.00      1.00         5
         yes       1.00      1.00      1.00         4
         you       1.00      1.00      1.00         4

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [6]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(best_y_te, best_preds)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_).plot(
    ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title(f'ISL Words — {best_name} | Acc: {best_acc*100:.1f}%', fontsize=13)
plt.tight_layout()
plt.savefig('models/word_confusion_matrix.png', dpi=150)
plt.show()
print('Saved: models/word_confusion_matrix.png')

# ── Per-class bar chart ───────────────────────────────────────────────────
pca = cm.diagonal() / cm.sum(axis=1)
fig2, ax2 = plt.subplots(figsize=(12, 5))
colors = ['#148A50' if a >= 0.9 else '#F06A2A' if a >= 0.7 else '#E74C3C' for a in pca]
bars   = ax2.bar(le.classes_, pca * 100, color=colors, edgecolor='white')
ax2.axhline(90, color='green',  linestyle='--', linewidth=1, alpha=0.7, label='90%')
ax2.axhline(70, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='70%')
ax2.set_ylim(0, 105);  ax2.legend()
ax2.set_xlabel('ISL Word');  ax2.set_ylabel('Accuracy (%)')
ax2.set_title(f'Per-Class Accuracy — ISL Words ({best_name})', fontsize=13)
for bar, acc in zip(bars, pca):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc*100:.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('models/word_per_class_accuracy.png', dpi=150)
plt.show()
print('Saved: models/word_per_class_accuracy.png')


Saved: models/word_confusion_matrix.png
Saved: models/word_per_class_accuracy.png


/var/folders/47/8ksx2fqj2cs06pgbq_30n2900000gn/T/ipykernel_49475/1094216542.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/47/8ksx2fqj2cs06pgbq_30n2900000gn/T/ipykernel_49475/1094216542.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Save all models ───────────────────────────────────────────────────────
import shutil

meta = dict(label_encoder=le, num_classes=NUM_CLASSES,
            feature_dim=63, max_frames=MAX_FRAMES, classes=list(le.classes_))

# SVM
with open('models/word_model_svm.pkl', 'wb') as f:
    pickle.dump({**meta, 'model': svm_pipe, 'model_type': 'SVM', 'accuracy': svm_acc}, f)
print(f'✅ SVM  → models/word_model_svm.pkl   ({svm_acc*100:.1f}%)')

# MLP
with open('models/word_model_mlp.pkl', 'wb') as f:
    pickle.dump({**meta, 'model': mlp_pipe, 'model_type': 'MLP', 'accuracy': mlp_acc}, f)
print(f'✅ MLP  → models/word_model_mlp.pkl   ({mlp_acc*100:.1f}%)')

# LSTM
torch.save({
    **meta,
    'model_state': best_state,
    'model_type': 'LSTM',
    'accuracy': lstm_acc,
    'arch': {'input_dim': 63, 'hidden': 256, 'layers': 2,
             'num_classes': NUM_CLASSES, 'dropout': 0.3},
}, 'models/word_model_lstm.pt')
print(f'✅ LSTM → models/word_model_lstm.pt   ({lstm_acc*100:.1f}%)')

# Default = best model
if best_name == 'LSTM':
    shutil.copy('models/word_model_lstm.pt', 'models/word_model.pt')
    print(f'\n🎯 Default → LSTM (models/word_model.pt)')
else:
    src = f'models/word_model_{best_name.lower()}.pkl'
    shutil.copy(src, 'models/word_model.pkl')
    print(f'\n🎯 Default → {best_name} (models/word_model.pkl)')

print(f'\n   Classes  : {list(le.classes_)}')
print(f'   Frames   : {MAX_FRAMES} per sample')
print(f'   Features : 63-dim per frame  (21 landmarks × xyz)')
print('\n▶ Next: run Step 3 to start the FastAPI server!')


✅ SVM  → models/word_model_svm.pkl   (94.4%)
✅ MLP  → models/word_model_mlp.pkl   (83.3%)
✅ LSTM → models/word_model_lstm.pt   (100.0%)

🎯 Default → LSTM (models/word_model.pt)

   Classes  : [np.str_('friend'), np.str_('hello'), np.str_('help'), np.str_('home'), np.str_('i_love_you'), np.str_('no'), np.str_('please'), np.str_('thank_you'), np.str_('yes'), np.str_('you')]
   Frames   : 60 per sample
   Features : 63-dim per frame  (21 landmarks × xyz)

▶ Next: run Step 3 to start the FastAPI server!


In [8]:
SERVER_CODE = '''
import pickle, torch, torch.nn as nn, numpy as np
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List
import uvicorn

app = FastAPI(title="ISL Word API")
app.add_middleware(CORSMiddleware,
    allow_origins=["http://localhost:5173","http://localhost:3000","http://127.0.0.1:5173"],
    allow_methods=["*"], allow_headers=["*"])

# ── Load models ──────────────────────────────────────────────────────────
with open("models/word_model_svm.pkl","rb") as f: svm_data = pickle.load(f)
with open("models/word_model_mlp.pkl","rb") as f: mlp_data = pickle.load(f)

class WordLSTM(nn.Module):
    def __init__(self, input_dim=63, hidden=256, layers=2, num_classes=10, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.attn = nn.Linear(hidden*2, 1)
        self.head = nn.Sequential(nn.Linear(hidden*2,128), nn.ReLU(),
                                   nn.Dropout(0.3), nn.Linear(128,num_classes))
    def forward(self, x):
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        return self.head((w * out).sum(dim=1))

ckpt       = torch.load("models/word_model_lstm.pt", map_location="cpu")
lstm_model = WordLSTM(**ckpt["arch"])
lstm_model.load_state_dict(ckpt["model_state"])
lstm_model.eval()
le         = ckpt["label_encoder"]
MAX_FRAMES = ckpt["max_frames"]
print(f"✅ Models loaded | Classes: {list(le.classes_)}")

# ── Schema ───────────────────────────────────────────────────────────────
class KeypointSequence(BaseModel):
    keypoints: List[List[float]]   # (T, 63)
    model: str = "lstm"            # "svm" | "mlp" | "lstm"

@app.post("/predict-word")
async def predict_word(req: KeypointSequence):
    arr = np.array(req.keypoints, dtype=np.float32)
    if len(arr) >= MAX_FRAMES:
        arr = arr[:MAX_FRAMES]
    else:
        pad = np.zeros((MAX_FRAMES - len(arr), 63), dtype=np.float32)
        arr = np.concatenate([arr, pad], axis=0)

    m = req.model.lower()
    if m in ("svm", "mlp"):
        pipe      = svm_data["model"] if m == "svm" else mlp_data["model"]
        flat      = arr.flatten().reshape(1, -1)
        idx       = pipe.predict(flat)[0]
        proba     = pipe.predict_proba(flat)[0]
        predicted = le.inverse_transform([idx])[0]
        confidence= float(proba[idx])
        top3      = sorted(zip(le.classes_, proba), key=lambda x: -x[1])[:3]
    else:
        t = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            proba = torch.softmax(lstm_model(t)[0], dim=-1).numpy()
        idx       = int(proba.argmax())
        predicted = le.inverse_transform([idx])[0]
        confidence= float(proba[idx])
        top3      = sorted(zip(le.classes_, proba.tolist()), key=lambda x: -x[1])[:3]

    return {
        "predicted":  predicted,
        "confidence": round(confidence, 4),
        "top3": [{"word": w, "prob": round(float(p), 4)} for w, p in top3],
        "model_used": m,
    }

@app.get("/classes")
def get_classes():
    return {"classes": list(le.classes_), "max_frames": MAX_FRAMES}

@app.get("/health")
def health():
    return {"status": "ok"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8001)
'''

with open('word_server.py', 'w') as f:
    f.write(SERVER_CODE)

print('✅  word_server.py written.')
print('   Run in terminal:  python word_server.py')
print('   Swagger docs  :   http://localhost:8001/docs')


✅  word_server.py written.
   Run in terminal:  python word_server.py
   Swagger docs  :   http://localhost:8001/docs
